# E-Commerce Customer Churn Prediction
**AIE322 — Machine Learning Assignment**

**Deliverable:** `LastName_FirstName_AIE322_Ecommerce_Churn.ipynb`

This notebook builds a leakage-free binary classification workflow for predicting `Churn` (0 = retained, 1 = churned). It covers EDA, reproducible splitting, preprocessing with `ColumnTransformer` and `Pipeline`, bias–variance, Logistic Regression regularization, Bagging/Pasting, AdaBoost, and hard/soft voting.

## Before running
1. Download the dataset from Kaggle.
2. Place `E Commerce Dataset.xlsx` or a CSV copy in the same folder as this notebook.
3. If the file has a different name or is in another folder, update `DATA_PATH` in the next cell.
4. Run all cells from top to bottom.

> The notebook deliberately fits imputers, encoders, and scalers using the training data only. The test set is held out until evaluation.

In [ ]:
# 1. Imports and reproducibility
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, BaggingClassifier, AdaBoostClassifier,
    VotingClassifier
)
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")

# Update this path if your dataset has a different name/location.
DATA_PATH = None  # Example: "E Commerce Dataset.xlsx"

## Part 1 — Load, clean, inspect, and explore the data
The common Kaggle workbook is named `E Commerce Dataset.xlsx`. This cell also accepts a CSV file. If `DATA_PATH` is `None`, the notebook searches the current folder for a matching Excel or CSV file.

In [ ]:
# 2. Load dataset
if DATA_PATH is None:
    candidates = (
        list(Path(".").glob("*E*Commerce*.xlsx")) +
        list(Path(".").glob("*E*Commerce*.xls")) +
        list(Path(".").glob("*churn*.csv")) +
        list(Path(".").glob("*Churn*.csv"))
    )
    if not candidates:
        candidates = list(Path(".").glob("*.xlsx")) + list(Path(".").glob("*.csv"))
    if not candidates:
        raise FileNotFoundError(
            "Dataset not found. Put the Excel/CSV file beside this notebook "
            "or set DATA_PATH to its exact path."
        )
    DATA_PATH = str(candidates[0])

path = Path(DATA_PATH)
if path.suffix.lower() in [".xlsx", ".xls"]:
    df = pd.read_excel(path)
elif path.suffix.lower() == ".csv":
    df = pd.read_csv(path)
else:
    raise ValueError("Unsupported file type. Use .xlsx, .xls, or .csv.")

print("Loaded:", path.name)
print("Shape:", df.shape)
display(df.head())

In [ ]:
# 3. Initial inspection and cleaning
df.columns = df.columns.astype(str).str.strip()
print("Column names:", df.columns.tolist())
print("\nData types and non-null counts:")
df.info()

print("\nMissing values (largest first):")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))

duplicate_count = df.duplicated().sum()
print("Exact duplicate rows:", duplicate_count)
df = df.drop_duplicates().copy()
print("Shape after removing exact duplicate rows:", df.shape)

if "Churn" not in df.columns:
    raise KeyError("The target column 'Churn' was not found. Check the dataset column names.")

# CustomerID is an identifier, not a meaningful customer-behaviour predictor.
if "CustomerID" in df.columns:
    df = df.drop(columns=["CustomerID"])

# Ensure the target is numeric and remove rows where the target is missing/invalid.
df["Churn"] = pd.to_numeric(df["Churn"], errors="coerce")
df = df.dropna(subset=["Churn"]).copy()
df = df[df["Churn"].isin([0, 1])].copy()
df["Churn"] = df["Churn"].astype(int)

print("\nTarget distribution:")
display(df["Churn"].value_counts().rename(index={0:"Retained (0)", 1:"Churned (1)"}).to_frame("count"))

### Outlier inspection
Outliers are flagged using the 1.5×IQR rule for numerical predictors. They are **not automatically deleted**: unusually high spending or order counts can be genuine customer behaviour and may help predict churn. Investigate values and correct only demonstrable data errors.

In [ ]:
# 4. Numerical/categorical columns and outlier counts
feature_cols = [c for c in df.columns if c != "Churn"]
num_cols = df[feature_cols].select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in feature_cols if c not in num_cols]

print("Numerical columns:", num_cols)
print("Categorical columns:", cat_cols)

outlier_rows = []
for col in num_cols:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    count = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_rows.append([col, lower, upper, int(count)])
outlier_summary = pd.DataFrame(
    outlier_rows, columns=["feature", "IQR_lower_bound", "IQR_upper_bound", "outlier_count"]
)
display(outlier_summary.sort_values("outlier_count", ascending=False))

### Exploratory Data Analysis (EDA)

In [ ]:
# 5. Class balance
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Churn")
plt.title("Customer Churn Class Balance")
plt.xlabel("Churn (0 = Retained, 1 = Churned)")
plt.ylabel("Number of customers")
plt.tight_layout()
plt.show()

# Numerical feature distributions (only those present in this dataset)
requested_num = ["Tenure", "WarehouseToHome", "CashbackAmount", "DaySinceLastOrder"]
available_num = [c for c in requested_num if c in df.columns]
if available_num:
    df[available_num].hist(figsize=(12, 8), bins=25, edgecolor="black")
    plt.suptitle("Distributions of Key Numerical Features")
    plt.tight_layout()
    plt.show()

# Numerical features by churn status
for col in available_num:
    plt.figure(figsize=(6, 4))
    sns.boxplot(data=df, x="Churn", y=col)
    plt.title(f"{col} by Churn")
    plt.tight_layout()
    plt.show()

# Categorical feature counts, coloured by churn
requested_cat = ["PreferredLoginDevice", "PreferredPaymentMode", "PreferedOrderCat"]
available_cat = [c for c in requested_cat if c in df.columns]
for col in available_cat:
    plt.figure(figsize=(10, 4))
    sns.countplot(data=df, x=col, hue="Churn")
    plt.title(f"{col} vs Churn")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

### Analytical question: Why is `random_state` essential?
`train_test_split` uses randomness when selecting which observations go into the training and test sets. Setting `random_state=42` fixes the pseudo-random sequence, so the same rows are assigned to each split every time the notebook is rerun (assuming the data and library behaviour are unchanged). This makes results reproducible, comparable, and easier to debug. The value 42 is conventional; any fixed integer would work. `stratify=y` additionally preserves approximately the same churn-class proportions in both subsets.

## Reproducible train/test split

In [ ]:
# 6. Separate predictors and target, then split BEFORE fitting preprocessing.
X = df.drop(columns=["Churn"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("\nTraining churn proportions:")
display(y_train.value_counts(normalize=True).sort_index().rename("proportion"))
print("\nTesting churn proportions:")
display(y_test.value_counts(normalize=True).sort_index().rename("proportion"))

## Part 2 — Preprocessing pipelines
The numerical branch imputes missing values with the training median and scales values to the [0, 1] range. The categorical branch fills missing values with the most frequent training category and one-hot encodes categories. `handle_unknown="ignore"` prevents an error if the test set contains a category not observed during training.

In [ ]:
# 7. Build preprocessing pipelines
# Compatibility across scikit-learn versions for OneHotEncoder's sparse argument.
try:
    one_hot = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
except TypeError:
    one_hot = OneHotEncoder(handle_unknown="ignore", sparse=True)

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", MinMaxScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", one_hot)
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, num_cols),
    ("cat", categorical_pipeline, cat_cols)
])

### Attribute inspection: learned medians
Fit the numerical imputer on `X_train` only and inspect `.statistics_`. This attribute contains the learned statistic for each input feature for all supported imputation strategies (median, mean, most-frequent, or constant). There is no general `.median_` attribute because the learned statistic is strategy-dependent; `.statistics_` provides one consistent interface.

In [ ]:
# 8. Inspect median values learned from training data only
median_imputer = SimpleImputer(strategy="median")
median_imputer.fit(X_train[num_cols])
median_values = pd.Series(median_imputer.statistics_, index=num_cols, name="learned_median")
display(median_values.to_frame())

### Sparse matrix inspection
One-hot encoding creates many indicator columns, most of which are zero for any single row. A sparse matrix stores only non-zero entries and their positions, reducing memory use compared with a dense NumPy array. This is particularly useful for high-cardinality categorical features. The default encoder output is sparse unless configured otherwise.

In [ ]:
# 9. Inspect OneHotEncoder output type
if cat_cols:
    cat_imputer = SimpleImputer(strategy="most_frequent")
    X_train_cat_imputed = cat_imputer.fit_transform(X_train[cat_cols])
    try:
        inspection_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        inspection_encoder = OneHotEncoder(handle_unknown="ignore", sparse=True)
    X_train_cat_encoded = inspection_encoder.fit_transform(X_train_cat_imputed)
    print("Encoded output type:", type(X_train_cat_encoded))
    print("Encoded shape:", X_train_cat_encoded.shape)
    print("Stored non-zero values:", X_train_cat_encoded.nnz if hasattr(X_train_cat_encoded, "nnz") else "N/A")
else:
    print("No categorical columns were detected.")

### Execution flow: what happens in `pipeline.fit(X_train, y_train)`?
1. The `Pipeline` calls `fit_transform(X_train, y_train)` on its preprocessing step (`ColumnTransformer`).
2. `ColumnTransformer` sends the selected numerical columns to the numerical pipeline and the categorical columns to the categorical pipeline.
3. Each branch fits its imputer on the training subset, learns the required replacement values, and transforms missing entries. The numerical branch then learns the scaling ranges and scales the data. The categorical branch learns the observed categories and creates one-hot columns.
4. The transformed outputs are concatenated into one feature matrix.
5. The final estimator receives that transformed training matrix and `y_train`, and its `fit` method learns the prediction model.
6. During `pipeline.predict(X_test)`, the already-fitted transformers only call `transform`; they do **not** relearn medians, ranges, or categories from the test set. The final estimator then predicts using the transformed test matrix.

Transformers learn how to prepare features; the final estimator learns the relationship between prepared features and the target. Keeping them in one pipeline helps prevent preprocessing leakage and ensures identical transformations at training and prediction time.

## Part 3 — Model complexity, bias–variance, and regularization

### Underfitting vs. overfitting
A shallow decision tree (`max_depth=1`) is a high-bias baseline: it has very limited ability to represent complex patterns. A deeper/unrestricted tree can fit more detailed patterns, but may also memorize noise.

- **Underfitting:** both training and test performance are poor; the model is too simple to capture the signal.
- **Overfitting:** training performance is much better than test performance; the model has learned training-specific noise or details that do not generalize.

In [ ]:
# 10. Evaluate models with a consistent helper
def evaluate_model(name, model, Xtr, ytr, Xte, yte):
    model.fit(Xtr, ytr)
    train_pred = model.predict(Xtr)
    test_pred = model.predict(Xte)
    result = {
        "Model": name,
        "Train Accuracy": accuracy_score(ytr, train_pred),
        "Test Accuracy": accuracy_score(yte, test_pred),
        "Test Precision": precision_score(yte, test_pred, zero_division=0),
        "Test Recall": recall_score(yte, test_pred, zero_division=0),
        "Test F1": f1_score(yte, test_pred, zero_division=0)
    }
    return result, model

shallow_tree = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE))
])
deep_tree = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(random_state=RANDOM_STATE))
])

bias_results = []
r, shallow_tree = evaluate_model("Shallow Decision Tree (depth=1)", shallow_tree,
                                  X_train, y_train, X_test, y_test)
bias_results.append(r)
r, deep_tree = evaluate_model("Unrestricted Decision Tree", deep_tree,
                              X_train, y_train, X_test, y_test)
bias_results.append(r)

bias_variance_results = pd.DataFrame(bias_results)
display(bias_variance_results)

### Analytical question: what happens to bias and variance as complexity increases?
As model complexity increases, **bias generally decreases** because the model can represent more complicated relationships. **Variance generally increases** because the model becomes more sensitive to the particular training sample. A very simple model may underfit (high bias, low variance); a very complex model may overfit (low bias, high variance). The goal is to choose a complexity that generalizes well to unseen data.

### If the model underfits, why increase complexity rather than only add training examples?
Underfitting means the model's hypothesis class is too restricted to represent the underlying relationship. Increasing complexity (for example, allowing a deeper tree) gives the model additional splits and decision boundaries, so it can capture patterns it previously could not express. More examples can reduce uncertainty and improve estimates, but they do not usually fix a model that is fundamentally too simple: the same restricted model may continue to underfit even with a much larger dataset. More data is especially useful for reducing variance in an overfit/high-variance model.

### Logistic Regression: Ridge (L2), Lasso (L1), and Elastic Net
In scikit-learn Logistic Regression, smaller `C` means stronger regularization; larger `C` means weaker regularization. L2 shrinks coefficients toward zero but usually does not make them exactly zero. L1 can set some coefficients exactly to zero, performing feature selection. Elastic Net combines L1 and L2; `l1_ratio=0.5` gives a 50/50 mixture. The `saga` solver supports all three penalties.

In [ ]:
# 11. Compare regularized Logistic Regression models
logistic_configs = [
    ("Logistic Regression L2 (Ridge)", {"penalty": "l2"}),
    ("Logistic Regression L1 (Lasso)", {"penalty": "l1"}),
    ("Logistic Regression Elastic Net", {"penalty": "elasticnet", "l1_ratio": 0.5})
]

logistic_results = []
logistic_models = {}

for name, params in logistic_configs:
    model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            solver="saga", max_iter=2000, C=1.0,
            random_state=RANDOM_STATE, **params
        ))
    ])
    result, fitted = evaluate_model(name, model, X_train, y_train, X_test, y_test)
    logistic_results.append(result)
    logistic_models[name] = fitted

logistic_results_df = pd.DataFrame(logistic_results)
display(logistic_results_df)

**Interpretation:** compare the test scores, not only training scores. The best penalty is the one that generalizes best for this dataset and split. L1 may produce a sparser model; L2 often gives stable shrinkage when many features contribute; Elastic Net can balance sparsity and coefficient stability. The exact winning configuration must be determined from the displayed results rather than assumed in advance.

## Part 4 — Ensemble learning

### Bagging vs. Pasting and Out-of-Bag evaluation
Bagging trains multiple base estimators on bootstrap samples (sampling rows **with replacement**) and aggregates their predictions. Pasting trains estimators on random subsets sampled **without replacement**. Thus, `bootstrap=False` in `BaggingClassifier` implements Pasting.

`n_jobs=-1` tells scikit-learn to use all available CPU cores for supported parallel work. This can reduce runtime, although actual speed depends on the machine and workload.

With bootstrap sampling, some training rows are left out of a particular estimator's bootstrap sample. Those out-of-bag (OOB) rows can be used to estimate generalization performance without creating a separate validation split for each tree. OOB evaluation is computationally efficient and makes use of training observations that were not used to fit each corresponding estimator.

In [ ]:
# 12. Transform features once using preprocessing fitted on training data only.
# This is safe: fit_transform is applied to training data; test uses transform only.
preprocessor_for_ensembles = preprocessor
X_train_transformed = preprocessor_for_ensembles.fit_transform(X_train)
X_test_transformed = preprocessor_for_ensembles.transform(X_test)

# Compatibility with scikit-learn versions that renamed base_estimator to estimator.
tree_base = DecisionTreeClassifier(random_state=RANDOM_STATE)
try:
    bagging_model = BaggingClassifier(
        estimator=tree_base, n_estimators=100, max_samples=1.0,
        bootstrap=True, oob_score=True, n_jobs=-1,
        random_state=RANDOM_STATE
    )
    pasting_model = BaggingClassifier(
        estimator=tree_base, n_estimators=100, max_samples=1.0,
        bootstrap=False, n_jobs=-1,
        random_state=RANDOM_STATE
    )
except TypeError:
    bagging_model = BaggingClassifier(
        base_estimator=tree_base, n_estimators=100, max_samples=1.0,
        bootstrap=True, oob_score=True, n_jobs=-1,
        random_state=RANDOM_STATE
    )
    pasting_model = BaggingClassifier(
        base_estimator=tree_base, n_estimators=100, max_samples=1.0,
        bootstrap=False, n_jobs=-1,
        random_state=RANDOM_STATE
    )

bagging_model.fit(X_train_transformed, y_train)
pasting_model.fit(X_train_transformed, y_train)

bagging_test_pred = bagging_model.predict(X_test_transformed)
pasting_test_pred = pasting_model.predict(X_test_transformed)

bagging_test_accuracy = accuracy_score(y_test, bagging_test_pred)
pasting_test_accuracy = accuracy_score(y_test, pasting_test_pred)

print("Bagging OOB accuracy:", bagging_model.oob_score_)
print("Bagging test accuracy:", bagging_test_accuracy)
print("Pasting test accuracy:", pasting_test_accuracy)

**OOB vs. test accuracy:** OOB accuracy is an internal estimate calculated from training observations omitted from each estimator's bootstrap sample. Test accuracy is calculated on the separate held-out test set. They need not be identical because they are based on different prediction procedures and sample sets. A close result is reassuring, but the test set remains the final independent evaluation.

### AdaBoost
AdaBoost trains weak learners sequentially. After each iteration, it increases the relative weights of misclassified training instances and decreases the relative influence of correctly classified instances. The next weak learner therefore focuses more strongly on examples that previous learners found difficult. The learners are combined using weights related to their performance.

In [ ]:
# 13. AdaBoost classifier
ada_base = DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE)
try:
    ada_model = AdaBoostClassifier(
        estimator=ada_base, n_estimators=100, learning_rate=0.5,
        random_state=RANDOM_STATE
    )
except TypeError:
    ada_model = AdaBoostClassifier(
        base_estimator=ada_base, n_estimators=100, learning_rate=0.5,
        random_state=RANDOM_STATE
    )

ada_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", ada_model)
])
ada_result, ada_pipeline = evaluate_model(
    "AdaBoost", ada_pipeline, X_train, y_train, X_test, y_test
)
display(pd.DataFrame([ada_result]))

ada_pred = ada_pipeline.predict(X_test)
print(classification_report(y_test, ada_pred, target_names=["Retained (0)", "Churned (1)"],
                            zero_division=0))

### Voting Classifiers: Hard vs. Soft voting
The ensemble combines three different model families: Logistic Regression, Random Forest, and Support Vector Classifier.

- **Hard voting:** each model casts a class-label vote; the class receiving the majority of votes is selected.
- **Soft voting:** each model contributes class probabilities; the probabilities are averaged (or weighted) and the class with the highest average probability is selected. Soft voting requires probability estimates, hence `SVC(probability=True)`.

Soft voting can use confidence information, but it is not guaranteed to outperform hard voting. The component models should ideally have reasonably calibrated probabilities.

In [ ]:
# 14. Hard and soft voting
def make_voting_pipeline(voting_method):
    voter = VotingClassifier(
        estimators=[
            ("lr", LogisticRegression(
                solver="saga", max_iter=2000, penalty="l2",
                random_state=RANDOM_STATE
            )),
            ("rf", RandomForestClassifier(
                n_estimators=200, random_state=RANDOM_STATE,
                n_jobs=-1, class_weight=None
            )),
            ("svc", SVC(
                probability=True, random_state=RANDOM_STATE
            ))
        ],
        voting=voting_method,
        n_jobs=-1
    )
    return Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", voter)
    ])

voting_results = []
voting_models = {}

for method in ["hard", "soft"]:
    model = make_voting_pipeline(method)
    result, fitted = evaluate_model(
        f"Voting ({method})", model, X_train, y_train, X_test, y_test
    )
    voting_results.append(result)
    voting_models[method] = fitted

voting_results_df = pd.DataFrame(voting_results)
display(voting_results_df)

## Final model comparison and conclusion

In [ ]:
# 15. Compare the main models
all_results = (
    bias_variance_results
    ._append(logistic_results_df, ignore_index=True)
    ._append(pd.DataFrame([{
        "Model": "Bagging",
        "Train Accuracy": accuracy_score(y_train, bagging_model.predict(X_train_transformed)),
        "Test Accuracy": bagging_test_accuracy,
        "Test Precision": precision_score(y_test, bagging_test_pred, zero_division=0),
        "Test Recall": recall_score(y_test, bagging_test_pred, zero_division=0),
        "Test F1": f1_score(y_test, bagging_test_pred, zero_division=0)
    }]), ignore_index=True)
    ._append(pd.DataFrame([{
        "Model": "Pasting",
        "Train Accuracy": accuracy_score(y_train, pasting_model.predict(X_train_transformed)),
        "Test Accuracy": pasting_test_accuracy,
        "Test Precision": precision_score(y_test, pasting_test_pred, zero_division=0),
        "Test Recall": recall_score(y_test, pasting_test_pred, zero_division=0),
        "Test F1": f1_score(y_test, pasting_test_pred, zero_division=0)
    }]), ignore_index=True)
    ._append(pd.DataFrame([ada_result]), ignore_index=True)
    ._append(voting_results_df, ignore_index=True)
)
display(all_results.sort_values("Test F1", ascending=False).reset_index(drop=True))

best_row = all_results.sort_values("Test F1", ascending=False).iloc[0]
print("Best model by test F1:", best_row["Model"])
print("Test F1:", round(best_row["Test F1"], 4))

In [ ]:
# 16. Confusion matrix for the best model by test F1
best_name = best_row["Model"]
if best_name == "Bagging":
    best_pred = bagging_test_pred
elif best_name == "Pasting":
    best_pred = pasting_test_pred
elif best_name == "AdaBoost":
    best_pred = ada_pred
elif best_name == "Voting (hard)":
    best_pred = voting_models["hard"].predict(X_test)
elif best_name == "Voting (soft)":
    best_pred = voting_models["soft"].predict(X_test)
elif best_name in logistic_models:
    best_pred = logistic_models[best_name].predict(X_test)
elif best_name == "Shallow Decision Tree (depth=1)":
    best_pred = shallow_tree.predict(X_test)
else:
    best_pred = deep_tree.predict(X_test)

ConfusionMatrixDisplay.from_predictions(
    y_test, best_pred, display_labels=["Retained (0)", "Churned (1)"],
    cmap="Blues"
)
plt.title(f"Confusion Matrix — {best_name}")
plt.tight_layout()
plt.show()

print("Final classification report:")
print(classification_report(y_test, best_pred,
                            target_names=["Retained (0)", "Churned (1)"],
                            zero_division=0))

### Conclusion template
After running the notebook, use the generated tables and plots to complete the following evidence-based summary:

1. **Class balance:** state whether the target is balanced or imbalanced and report the churn proportion.
2. **EDA:** mention which numerical and categorical variables appear to differ across churn groups. Describe visible patterns rather than claiming causation.
3. **Bias–variance:** compare shallow and unrestricted tree training/test accuracy. A large train–test gap suggests overfitting; low scores on both suggest underfitting.
4. **Regularization:** identify which of L1, L2, and Elastic Net achieved the strongest test F1/recall, and discuss the trade-off.
5. **Ensembles:** compare Bagging, Pasting, AdaBoost, and both voting approaches using the test metrics. Report Bagging's OOB score alongside test accuracy.
6. **Business interpretation:** for a churn-retention use case, recall for class 1 is important because false negatives are churners the business fails to identify. Precision is also relevant because retention offers have a cost. The appropriate operating point depends on the cost of missed churners versus unnecessary retention campaigns.

**Important:** Do not write that a model is best based on accuracy alone. Consider precision, recall, F1, and the business cost of errors.

## Reproducibility checklist
- [ ] Dataset file is present and `DATA_PATH` points to it.
- [ ] Notebook runs from the first cell to the last without manual hidden state.
- [ ] `random_state=42` is used for the split and stochastic estimators.
- [ ] Preprocessing is fitted only on training data.
- [ ] All plots, metric tables, and model outputs are visible after running all cells.
- [ ] Save the completed notebook using the required filename: `LastName_FirstName_AIE322_Ecommerce_Churn.ipynb`.